# 01 — Data Cleaning

This notebook cleans the four raw input datasets independently and saves the
cleaned versions to `data/interim/`. Run this notebook first.

**Inputs (from `data/raw/`):**
1. `2024hourlymeteo.csv` — Hourly Barcelona meteorology
2. `consum_electric/hora_poblenou.csv` — Hourly electricity from Poblenou campus
3. `aules/2024Tanger.xlsx` — 2024 classroom schedule for Tanger building
4. `Calendari-UPF-2024.csv` — Official UPF academic calendar

**Outputs (to `data/interim/`):**
- `clima_limpio.csv`
- `energia_limpio.csv`
- `ocupacio_aules_horaria.csv`
- (calendar is already clean; no transformation needed)


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RAW = Path('../data/raw')
DATA_INTERIM = Path('../data/interim')
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Reading from: {DATA_RAW.resolve()}")
print(f"Writing to:   {DATA_INTERIM.resolve()}")


c:\Users\jordi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Reading from: C:\Users\jordi\TFG\data\raw
Writing to:   C:\Users\jordi\TFG\data\interim


## 1. Meteorological data

In [2]:
# Load raw hourly weather
df_clima = pd.read_csv(DATA_RAW / '2024hourlymeteo.csv')

# Build a single Timestamp column from year/month/day/hour
df_clima['Timestamp'] = pd.to_datetime(df_clima[['year', 'month', 'day', 'hour']])

# Keep only the columns we need + rename for consistency
df_clima_limpio = df_clima[['Timestamp', 'temp', 'prcp']].rename(
    columns={'temp': 'Temperatura', 'prcp': 'Lluvia'}
)

# Sanity checks
print(f"Rows: {len(df_clima_limpio):,}  (expected 8784 for 2024 leap year)")
print(f"Date range: {df_clima_limpio['Timestamp'].min()} → {df_clima_limpio['Timestamp'].max()}")
print(f"NaN per column:\n{df_clima_limpio.isna().sum()}")

df_clima_limpio.to_csv(DATA_INTERIM / 'clima_limpio.csv', index=False)
df_clima_limpio.head()


Rows: 8,784  (expected 8784 for 2024 leap year)
Date range: 2024-01-01 00:00:00 → 2024-12-31 23:00:00
NaN per column:
Timestamp        0
Temperatura      0
Lluvia         251
dtype: int64


,Timestamp,Temperatura,Lluvia
0,2024-01-01 00:00:00,7.5,0.0
1,2024-01-01 01:00:00,8.0,0.0
2,2024-01-01 02:00:00,7.7,0.0
3,2024-01-01 03:00:00,7.5,0.0
4,2024-01-01 04:00:00,7.7,0.0


## 2. Electricity consumption (Poblenou Campus)

In [3]:
df_energia = pd.read_csv(
    DATA_RAW / 'consum_electric' / 'hora_poblenou.csv',
    sep=';'
)

columnas_utiles = ['Fecha y hora de la medida',
                   'Medida de la magnitud Activa Entrante (neta)']
df_energia_limpio = df_energia[columnas_utiles].copy()

# Rename 
df_energia_limpio.columns = ['Timestamp', 'Consumo_kWh']

# Parse timestamp handling hours ≥24 as next-day time
timestamps = df_energia_limpio['Timestamp'].astype(str).str.strip()

# Extract hour and detect invalid hours (≥24)
hour_pattern = r'(\d{2}):(\d{2}):(\d{2})$'
hours = timestamps.str.extract(hour_pattern)[0].astype(int)
mask_invalid_hour = hours >= 24

# Replace invalid hours with valid ones (e.g., 25:00:00 → 01:00:00)
timestamps = timestamps.str.replace(r'(\d{2}):(\d{2}):(\d{2})$', 
                                     lambda m: f"{int(m.group(1)) % 24:02d}:{m.group(2)}:{m.group(3)}", 
                                     regex=True)

df_energia_limpio['Timestamp'] = pd.to_datetime(timestamps, dayfirst=True) + pd.to_timedelta(mask_invalid_hour.astype(int), unit='d')

# Some rows might have comma as decimal separator
if df_energia_limpio['Consumo_kWh'].dtype == 'object':
    df_energia_limpio['Consumo_kWh'] = (
        df_energia_limpio['Consumo_kWh']
        .astype(str).str.replace(',', '.').astype(float)
    )

# Sanity checks
print(f"Rows: {len(df_energia_limpio):,}")
print(f"Date range: {df_energia_limpio['Timestamp'].min()} → {df_energia_limpio['Timestamp'].max()}")
print(f"Consumo stats:\n{df_energia_limpio['Consumo_kWh'].describe()}")

df_energia_limpio.to_csv(DATA_INTERIM / 'energia_limpio.csv', index=False)
df_energia_limpio.head()


Rows: 8,784
Date range: 2024-01-01 01:00:00 → 2025-01-01 00:00:00
Consumo stats:
count    8784.000000
mean      184.056580
std       108.999753
min         0.000000
25%        99.000000
50%       117.000000
75%       270.000000
max       471.000000
Name: Consumo_kWh, dtype: float64


,Timestamp,Consumo_kWh
0,2024-01-01 01:00:00,96
1,2024-01-01 02:00:00,96
2,2024-01-01 03:00:00,97
3,2024-01-01 04:00:00,96
4,2024-01-01 05:00:00,95


## 3. Classroom occupancy (Tanger building)

The Excel file lists every reservation for every classroom. We need to
aggregate to **hourly classroom occupancy** for the whole building.


In [4]:
TOTAL_AULAS = 94

# Load reservations
df_aulas = pd.read_excel(DATA_RAW / 'aules' / '2024Tanger.xlsx')

# Filter out rows with missing start or end dates/times
df_aulas = df_aulas.dropna(subset=['Data inicial', 'Hora inicial', 'Data final', 'Hora final'])

# Build Timestamp columns for the real Start and End of each class
df_aulas['Inici'] = pd.to_datetime(df_aulas['Data inicial'].astype(str) + ' ' + 
                                   df_aulas['Hora inicial'].astype(str), dayfirst=True)

df_aulas['Final'] = pd.to_datetime(df_aulas['Data final'].astype(str) + ' ' + 
                                   df_aulas['Hora final'].astype(str), dayfirst=True)

# Create a complete hourly grid for 2024 (8784 hours)
hourly_grid = pd.date_range('2024-01-01', '2024-12-31 23:00', freq='H')

# Calculate hourly occupancy based on time overlap
occupancy_list = []

for hour in hourly_grid:
    next_hour = hour + pd.Timedelta(hours=1)
    
    # A classroom is occupied if the reservation starts before the hour ends 
    # AND ends after the hour begins (handles fractional hours like 09:30 - 11:30)
    occupied_mask = (df_aulas['Inici'] < next_hour) & (df_aulas['Final'] > hour)
    
    # Count unique classrooms (Espai) occupied during this hour
    classrooms_occupied = df_aulas[occupied_mask]['Espai'].nunique()
    
    occupancy_list.append(classrooms_occupied)

# Build the final DataFrame
df_ocupacio = pd.DataFrame({
    'Timestamp': hourly_grid,
    'Aules_Ocupades': occupancy_list
})

# Build the percentage occupied
df_ocupacio['Ocupacio_Percent'] = (df_ocupacio['Aules_Ocupades'] / TOTAL_AULAS) * 100

# 8. Print summary metrics
print(f"Rows: {len(df_ocupacio):,}  (expected 8784)")
print(f"Max classrooms occupied at once: {df_ocupacio['Aules_Ocupades'].max()} / {TOTAL_AULAS}")
print(f"Mean occupancy: {df_ocupacio['Ocupacio_Percent'].mean():.1f}%")

# save to interim data for downstream modeling
df_ocupacio.to_csv(DATA_INTERIM / 'ocupacio_aules_horaria.csv', index=False)
df_ocupacio.head()

C:\Users\jordi\AppData\Local\Temp\ipykernel_28168\2906929312.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_grid = pd.date_range('2024-01-01', '2024-12-31 23:00', freq='H')


Rows: 8,784  (expected 8784)
Max classrooms occupied at once: 94 / 94
Mean occupancy: 13.5%


,Timestamp,Aules_Ocupades,Ocupacio_Percent
0,2024-01-01 00:00:00,0,0.0
1,2024-01-01 01:00:00,0,0.0
2,2024-01-01 02:00:00,0,0.0
3,2024-01-01 03:00:00,0,0.0
4,2024-01-01 04:00:00,0,0.0


## 4. Calendar — passthrough check

The calendar CSV is already clean. We just verify it loads properly and
confirm the day-type categories.


In [5]:
# 1. Leemos el archivo
df_calendari = pd.read_csv(
        DATA_RAW / 'Calendari-UPF-2024.csv',
    encoding='latin1'
)

# 2. Convertimos a datetime especificando el formato exacto Año-Mes-Día
df_calendari['fecha'] = pd.to_datetime(df_calendari['fecha'], format='%Y-%m-%d', errors='coerce')

# 3. Comprobaciones
print(f"Filas totales: {len(df_calendari)}")
print(f"Fechas nulas/rotas: {df_calendari['fecha'].isna().sum()}")
print(f"\nRango de fechas: {df_calendari['fecha'].min().date()} → {df_calendari['fecha'].max().date()}")
print(f"\nConteos por tipo de día:\n{df_calendari['tipus_dia'].value_counts()}")

df_calendari.head()

Filas totales: 366
Fechas nulas/rotas: 0

Rango de fechas: 2024-01-01 → 2024-12-31

Conteos por tipo de día:
tipus_dia
Classe           140
Altre             55
Dissabte          52
Diumenge          52
Avaluacio         30
Vacances          19
Festiu             8
No lectiu          6
La Benvinguda      4
Name: count, dtype: int64


,fecha,tipus_dia
0,2024-01-01,Vacances
1,2024-01-02,Vacances
2,2024-01-03,Vacances
3,2024-01-04,Vacances
4,2024-01-05,Vacances
